# Ablation 1 E2E RAG Evaluation

Runs Chunk-only versus Chunk+Metadata end-to-end with the same generator. Retrieval-only/raw FAISS diagnostics are kept separate from the main E2E comparison.

## 1. Editable Configuration

Set the branch to `ablation1` before merge, then switch it back to `main` after the changes land.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/PhuongThao-2005/TextMining.git"
REPO_BRANCH = "ablation1"
REPO_DIR = Path("/kaggle/working/TextMining")

FORCE_RECLONE = False
PULL_IF_EXISTS = True
INSTALL_DEPENDENCIES = True

CONFIG_NAMES = ["Embed-ChunkOnly-Dense", "Embed-ChunkMeta-Dense"]
RUN_VARIANTS = [
    {"name": "e2e-top10", "top_k": 10},
]
# Set True only if you intentionally want to rerun the earlier top-5 production setting.
RERUN_PREVIOUS_TOP5 = False
if RERUN_PREVIOUS_TOP5:
    RUN_VARIANTS.insert(0, {"name": "e2e-top5", "top_k": 5})

SMOKE_LIMIT = None  # set an integer such as 5 for a cheap test

QA_PATH = Path("/kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark/qa_final.jsonl")
CORPUS_PATH = Path("/kaggle/input/datasets/nguyenlethienlyy/text-mining-data-preprocessed/documents.jsonl")
INDEX_SOURCES = {
    "Embed-ChunkOnly-Dense": Path("/kaggle/input/datasets/kittrntunk/faisse-only-chunk/faiss_index"),
    "Embed-ChunkMeta-Dense": Path("/kaggle/input/datasets/kittrntunk/faiss-chunk-meta"),
}

RUNS_ROOT = Path("/kaggle/working/evaluation_runs/ablation1_e2e")
EXPORT_ROOT = Path("/kaggle/working/e2e_rag_outputs/ablation1_e2e")
RUNTIME_INDEX_ROOT = Path("/kaggle/working/runtime_indexes/ablation1_e2e")

USE_GPU_WHEN_AVAILABLE = True
CREATE_EXPORT_ZIP = True

# If empty, the model comes from Kaggle Secret LLM_BASE_MODEL.
GENERATION_MODEL_OVERRIDE = "gpt-4o-mini"

PREVIOUS_BAD_REFERENCE = {
    "label": "previous e2e top5 production run",
    "config_name": "Embed-ChunkMeta-Dense",
    "top_k": 5,
    "total_input": 500,
    "evaluated": 500,
    "recall@5": 0.1405,
    "token_f1": 0.3208,
    "rouge_l": 0.2953,
    "average_total_latency_ms": 3085.514,
}

## 2. Environment Setup

In [ ]:
import importlib.util
import json
import os
import platform
import shutil
import sqlite3
import subprocess
import sys
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_INPUT_ROOT.is_dir() and KAGGLE_WORKING_ROOT.is_dir()
print({"kaggle_environment": IS_KAGGLE, "python": platform.python_version()})

def _module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        return False

def _run(command, *, cwd=None, sensitive_values=()):
    completed = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode:
        detail = (completed.stderr or completed.stdout or "command failed")[-1200:]
        for secret in sensitive_values:
            if secret:
                detail = detail.replace(secret, "***")
        raise RuntimeError(detail)
    return completed.stdout.strip()

def valid_checkout(repo_dir):
    root = Path(repo_dir)
    return (root / ".git").is_dir() and (root / "configs" / "ablation_configs.yaml").exists() and (root / "src").exists()

if FORCE_RECLONE and REPO_DIR.exists():
    target = REPO_DIR.resolve()
    if target in {Path("/").resolve(), Path("/kaggle").resolve(), Path("/kaggle/working").resolve()}:
        raise RuntimeError("Refusing to remove a broad Kaggle directory.")
    shutil.rmtree(target)

if REPO_DIR.exists():
    if not valid_checkout(REPO_DIR):
        raise RuntimeError(f"Existing repository directory is not a valid checkout: {REPO_DIR}")
    _run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    if PULL_IF_EXISTS:
        _run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    _run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = _run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
os.chdir(REPO_DIR)
for import_root in (REPO_DIR, REPO_DIR / "src"):
    value = str(import_root)
    if value not in sys.path:
        sys.path.insert(0, value)
print({"repo": str(REPO_DIR), "branch": REPO_BRANCH, "commit": commit})

requirements = [("yaml", "PyYAML"), ("openai", "openai"), ("faiss", "faiss-cpu"), ("sentence_transformers", "sentence-transformers")]
missing = [package for module, package in requirements if not _module_available(module)]
if INSTALL_DEPENDENCIES and missing:
    _run([sys.executable, "-m", "pip", "install", *missing])
unresolved = [module for module, _ in requirements if not _module_available(module)]
print({"missing_after_setup": unresolved})
if unresolved:
    raise RuntimeError("Restart the Kaggle session and rerun from the first cell.")

## 3. Secrets And Inputs

In [ ]:
SECRET_MAPPING = {
    "LLM_BASE_URL": "LLM_BASE_URL",
    "LLM_API_KEY": "LLM_API_KEY",
    "LLM_BASE_MODEL": "LLM_BASE_MODEL",
}

def load_kaggle_secrets(secret_mapping):
    diagnostics = {}
    client = None
    for env_name, secret_name in secret_mapping.items():
        if os.environ.get(env_name):
            diagnostics[env_name] = "configured"
            continue
        if client is None:
            try:
                from kaggle_secrets import UserSecretsClient
                client = UserSecretsClient()
            except Exception:
                client = False
        value = None
        if client:
            try:
                value = client.get_secret(secret_name)
            except Exception:
                value = None
        if value:
            os.environ[env_name] = value
            diagnostics[env_name] = "configured"
        else:
            diagnostics[env_name] = "missing"
    return diagnostics

secret_status = load_kaggle_secrets(SECRET_MAPPING)
print(secret_status)

if GENERATION_MODEL_OVERRIDE:
    os.environ["LLM_BASE_MODEL"] = GENERATION_MODEL_OVERRIDE

required_inputs = {"qa": QA_PATH, "corpus": CORPUS_PATH}
for config_name, index_dir in INDEX_SOURCES.items():
    required_inputs[f"{config_name}.index"] = index_dir / "index.faiss"
    required_inputs[f"{config_name}.payloads"] = index_dir / "payloads.jsonl"
missing_inputs = {name: str(path) for name, path in required_inputs.items() if not Path(path).exists()}
print({"missing_inputs": missing_inputs})
if missing_inputs:
    raise FileNotFoundError(missing_inputs)

missing_generation = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_BASE_MODEL") if not os.environ.get(name)]
if missing_generation:
    raise RuntimeError("Missing generator secrets/env values: " + ", ".join(missing_generation))

## 4. Run E2E Ablation 1

In [ ]:
from copy import deepcopy

from scripts.run_ablation_config import (
    apply_runtime_path_overrides,
    load_ablation_configs,
    resolve_ablation_config,
    run_ablation_config,
    validate_ablation_config,
)
from evaluation.artifacts import load_run_artifacts
from evaluation.export import export_run_artifacts


def link_or_copy(src, dest):
    if dest.exists() or dest.is_symlink():
        return
    try:
        os.symlink(src, dest)
    except OSError:
        shutil.copy2(src, dest)


def prepare_runtime_index(config_name, source_dir):
    source = Path(source_dir)
    dest = RUNTIME_INDEX_ROOT / config_name
    dest.mkdir(parents=True, exist_ok=True)
    for filename in ["index.faiss", "payloads.jsonl", "id_map.json", "payload_offsets.pkl", "payloads_export.csv"]:
        src = source / filename
        if src.exists():
            link_or_copy(src, dest / filename)
    src_cache = source / "payload_cache.sqlite"
    dest_cache = dest / "payload_cache.sqlite"
    payloads_path = dest / "payloads.jsonl"
    if src_cache.exists() and payloads_path.exists() and not dest_cache.exists():
        shutil.copy2(src_cache, dest_cache)
        stat = payloads_path.stat()
        conn = sqlite3.connect(str(dest_cache))
        try:
            conn.execute("UPDATE meta SET value=? WHERE key='payload_size'", (str(stat.st_size),))
            conn.execute("UPDATE meta SET value=? WHERE key='payload_mtime_ns'", (str(stat.st_mtime_ns),))
            conn.commit()
        finally:
            conn.close()
    return {"index": dest / "index.faiss", "payloads": dest / "payloads.jsonl", "dir": dest}


def normalize_generation_numbers(config):
    generation = config.setdefault("generation", {})
    for key, default in {"max_output_tokens": 1024, "max_retries": 2}.items():
        value = generation.get(key, default)
        generation[key] = default if value in (None, "") else int(value)
    for key, default in {"timeout_seconds": 60.0, "temperature": 0.0, "top_p": 1.0}.items():
        value = generation.get(key, default)
        generation[key] = default if value in (None, "") else float(value)
    return config

configs = load_ablation_configs(REPO_DIR / "configs" / "ablation_configs.yaml")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

cuda_available = False
if _module_available("torch"):
    try:
        import torch
        cuda_available = bool(torch.cuda.is_available())
    except Exception:
        cuda_available = False
selected_device = "cuda" if USE_GPU_WHEN_AVAILABLE and cuda_available else "cpu"
print({"selected_device": selected_device})

run_records = []
for variant in RUN_VARIANTS:
    for config_name in CONFIG_NAMES:
        source_config = resolve_ablation_config(configs, config_name)
        validate_ablation_config(source_config, config_name=config_name)
        prepared = prepare_runtime_index(config_name, INDEX_SOURCES[config_name])
        resolved = apply_runtime_path_overrides(
            source_config,
            benchmark_source=QA_PATH,
            corpus_source=CORPUS_PATH,
            faiss_index_source=prepared["index"],
            faiss_payloads_source=prepared["payloads"],
            runs_root=RUNS_ROOT,
            selected_device=selected_device,
        )
        resolved = normalize_generation_numbers(resolved)
        resolved["retrieval"]["top_k"] = int(variant["top_k"])
        resolved["generation"]["model"] = os.environ["LLM_BASE_MODEL"]
        resolved["generation"]["model_env"] = "LLM_BASE_MODEL"
        resolved.setdefault("metadata", {})["ablation1_variant"] = variant["name"]
        resolved.setdefault("metadata", {})["kaggle_input_identities"] = {
            "benchmark": str(QA_PATH),
            "corpus": str(CORPUS_PATH),
            "faiss_index": str(INDEX_SOURCES[config_name] / "index.faiss"),
            "faiss_payloads": str(INDEX_SOURCES[config_name] / "payloads.jsonl"),
        }
        print({"running": config_name, "variant": variant["name"], "top_k": resolved["retrieval"]["top_k"]})
        outcome = run_ablation_config(
            config_name,
            config_file=REPO_DIR / "configs" / "ablation_configs.yaml",
            output_root=RUNS_ROOT,
            limit=SMOKE_LIMIT,
            project_root=REPO_DIR,
            resolved_config_override=resolved,
        )
        if outcome.status != "completed":
            raise RuntimeError(f"{config_name} {variant['name']} ended with {outcome.status}: {outcome.error}")
        artifacts = load_run_artifacts(outcome.output_dir, require_completed=True)
        export_result = export_run_artifacts(
            artifacts.run_dir,
            EXPORT_ROOT,
            create_zip=CREATE_EXPORT_ZIP,
            sensitive_values=(os.environ.get("LLM_API_KEY", ""),),
        )
        run_records.append({"config_name": config_name, "variant": variant["name"], "artifacts": artifacts, "export": export_result})
        print({"completed": config_name, "run_dir": str(artifacts.run_dir)})

print({"completed_runs": len(run_records)})

## 5. Compare Results

In [ ]:
def flatten_summary(record):
    artifacts = record["artifacts"]
    metrics = artifacts.metrics or {}
    overall = metrics.get("overall") or {}
    latency = artifacts.latency or {}
    total_latency = ((latency.get("stages") or {}).get("total") or {})
    return {
        "variant": record["variant"],
        "config_name": record["config_name"],
        "run_dir": str(artifacts.run_dir),
        "count": overall.get("count"),
        "token_f1": overall.get("token_f1"),
        "rouge_l": overall.get("rouge_l"),
        "exact_match": overall.get("exact_match"),
        "unanswerable_accuracy": overall.get("unanswerable_accuracy"),
        "context_recall@k": overall.get("context_recall@k"),
        "recall@5": overall.get("recall@5"),
        "recall@10": overall.get("recall@10"),
        "mrr@10": overall.get("mrr@10"),
        "ndcg@10": overall.get("ndcg@10"),
        "total_latency_mean_ms": total_latency.get("mean"),
        "metric_denominators": overall.get("metric_denominators"),
    }

rows = [flatten_summary(record) for record in run_records]
comparison_rows = [PREVIOUS_BAD_REFERENCE, *rows]
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(comparison_rows))
except Exception:
    print(json.dumps(comparison_rows, ensure_ascii=False, indent=2))

for row in rows:
    print("\n#", row["variant"], row["config_name"])
    print(Path(row["run_dir"], "report.md").read_text(encoding="utf-8")[:5000])

## 6. Optional Raw FAISS Retrieval Diagnostic

In [ ]:
RUN_RAW_FAISS_DIAGNOSTIC = False
if RUN_RAW_FAISS_DIAGNOSTIC:
    for config_name in CONFIG_NAMES:
        source_dir = INDEX_SOURCES[config_name]
        out_dir = Path("/kaggle/working/evaluation_runs/ablation1_raw_faiss") / config_name
        runtime_dir = RUNTIME_INDEX_ROOT / f"raw_{config_name}"
        command = [
            sys.executable,
            "scripts/evaluate_retrieval.py",
            "--qa-path", str(QA_PATH),
            "--store", "faiss",
            "--raw-faiss",
            "--index-dir", str(source_dir),
            "--runtime-index-dir", str(runtime_dir),
            "--out-dir", str(out_dir),
            "--top-k", "1", "5", "10",
            "--include-empty-ground-truth",
        ]
        print("Running", " ".join(command))
        _run(command, cwd=REPO_DIR)
        print((out_dir / "retrieval_report.md").read_text(encoding="utf-8")[:3000])
else:
    print("Raw FAISS diagnostic is disabled. Set RUN_RAW_FAISS_DIAGNOSTIC = True to run it.")